<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "659"
iso_country = "Saint Kitts and Nevis"
import_portfolio = "h3_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h3_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2017
gdp = 1.060e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Saint Kitts and Nevis': 

Number of trade records present in dataset: 7764
Number of partner countries present in dataset: 101
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$255.5m
Total weight within the dataset: 216.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,659,Saint Kitts and Nevis,842,USA,Import,Sea,440310,"440310 | Wood; in the rough,",4403,"4403 | Wood in the rough, w",44,44 | Wood and articles of,3131,"Wood in the rough, treated with paint, stains,...",Wood and Paper,30,LUM - Wood products,14192.775,NaN,14192.775,NaN,0.0,NaN,100568.584373,3176.0,1973.474296,198469.516244,['all'],['all'],503.293684,502.302892,502.416171,502.416171,502.852768,503.254511,519.568244,854.917613,552.86338,522.297819,524.414999,539.630925,510.770617,836.773299,580.923043,542.066542,532.622535,553.360166,502.744218,919.056464,690.192236,630.058204,590.321581,573.372781,494.922386,1015.384767,806.721173,760.238150,616.728073,608.945662,486.966963,1019.145501,955.720775,992.603294,669.660755,636.818353,478.946181,942.099279,898.038516,972.718615,748.723889,646.494106,471.469104,874.594874,900.784195,972.218813,756.94278,654.339766,463.594444,805.375677,820.751974,949.308716,785.181677,661.983772,3131
1,1,659,Saint Kitts and Nevis,842,USA,Import,Sea,930119,930119 | Military weapons; ot,9301,9301 | Military weapons; ot,93,93 | Arms and ammunition;,4472,"Military weapons, other than revolvers, pistol...",Other Manufacturing,41,OME - Machinery and equipment n.e.c.,69.526,NaN,69.526,NaN,0.0,NaN,0.389346,3176.0,1973.474296,0.768364,['all'],['all'],0.001948,0.001945,0.001945,0.001945,0.001947,0.001948,0.002011,0.003310,0.00214,0.002022,0.002030,0.002089,0.001977,0.003240,0.002249,0.002099,0.002062,0.002142,0.001946,0.003558,0.002672,0.002439,0.002285,0.002220,0.001916,0.003931,0.003123,0.002943,0.002388,0.002357,0.001885,0.003946,0.003700,0.003843,0.002593,0.002465,0.001854,0.003647,0.003477,0.003766,0.002899,0.002503,0.001825,0.003386,0.003487,0.003764,0.00293,0.002533,0.001795,0.003118,0.003177,0.003675,0.003040,0.002563,4472


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $75.4m (29.5%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
02 | Meat and edible meat  10354693.0   4681712.6      4.1      2.2
22 | Beverages, spirits a   9244013.0  26167543.2      3.6     12.1
21 | Miscellaneous edible   8047535.7   2191712.3      3.2      1.0
04 | Dairy produce; birds   7030230.5   4060948.4      2.8      1.9
19 | Preparations of cere   6332218.0   2509115.9      2.5      1.2
20 | Preparations of vege   5256097.4   3764575.4      2.1      1.7
07 | Vegetables and certa   4810141.9   6314875.8      1.9      2.9
03 | Fish and crustaceans   3985577.4    766121.0      1.6      0.4
16 | Meat, fish, crustace   3696134.5    812175.4      1.4      0.4
11 | Products of the mill   3213988.5   6744523.6      1.3      3.1
23 | Food industries, res   3118541.4   2310074.3      1.2      1.1
08 | Fruit and n

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $77.5m (30.4%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Food & Beverages  39571423.9  40998381.6     15.5     19.0
Agriculture       33990367.7  28983858.1     13.3     13.4
Fishing            3985577.4    766121.0      1.6      0.4



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $66.7m (26.1%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     32712413.3  18748537.7     12.8      8.7
OMT - Meat products n.e.c.     10973059.7   4741652.8      4.3      2.2
MIL - Dairy products            7443102.2   4168857.4      2.9      1.9
V_F - Vegetables, fruit, nuts   6978872.4   8145743.7      2.7      3.8
VOL - Vegetable oils and fats   2348447.5   1708632.2      0.9      0.8
CMT - Bovine meat prods         1961648.5    528962.4      0.8      0.2
OCR - Crops n.e.c.              1212380.6    444745.8      0.5      0.2
SGR - Sugar                     1184805.8   2722345.2      0.5      1.3
PDR - Paddy rice                1118814.3   1943380.5      0.4      0.9
OAP - Animal products n.e.c.     353488.9    221553.3      0.1      0.1
FSH - Fishing                    208745.9     47292.4      0.1      0.0

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 29.5% 
GTAP 26.1% 
EORA 30.4%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16227/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,659,Saint Kitts and Nevis,842,USA,Import,Sea,440310,"440310 | Wood; in the rough,",4403,"4403 | Wood in the rough, w",44,44 | Wood and articles of,3131,"Wood in the rough, treated with paint, stains,...",Wood and Paper,30,LUM - Wood products,14192.775,NaN,14192.775,NaN,0.0,NaN,100568.584373,3176.0,1973.474296,198469.516244,['all'],['all'],503.293684,502.302892,502.416171,502.416171,502.852768,503.254511,519.568244,854.917613,552.86338,522.297819,524.414999,539.630925,510.770617,836.773299,580.923043,542.066542,532.622535,553.360166,502.744218,919.056464,690.192236,630.058204,590.321581,573.372781,494.922386,1015.384767,806.721173,760.238150,616.728073,608.945662,486.966963,1019.145501,955.720775,992.603294,669.660755,636.818353,478.946181,942.099279,898.038516,972.718615,748.723889,646.494106,471.469104,874.594874,900.784195,972.218813,756.94278,654.339766,463.594444,805.375677,820.751974,949.308716,785.181677,661.983772,3131,NaN
1,1,659,Saint Kitts and Nevis,842,USA,Import,Sea,930119,930119 | Military weapons; ot,9301,9301 | Military weapons; ot,93,93 | Arms and ammunition;,4472,"Military weapons, other than revolvers, pistol...",Other Manufacturing,41,OME - Machinery and equipment n.e.c.,69.526,NaN,69.526,NaN,0.0,NaN,0.389346,3176.0,1973.474296,0.768364,['all'],['all'],0.001948,0.001945,0.001945,0.001945,0.001947,0.001948,0.002011,0.003310,0.00214,0.002022,0.002030,0.002089,0.001977,0.003240,0.002249,0.002099,0.002062,0.002142,0.001946,0.003558,0.002672,0.002439,0.002285,0.002220,0.001916,0.003931,0.003123,0.002943,0.002388,0.002357,0.001885,0.003946,0.003700,0.003843,0.002593,0.002465,0.001854,0.003647,0.003477,0.003766,0.002899,0.002503,0.001825,0.003386,0.003487,0.003764,0.00293,0.002533,0.001795,0.003118,0.003177,0.003675,0.003040,0.002563,4472,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $48.3m of $255.5m (18.9%)
Volume: 36.7 of 216.3 million kilograms (17.0%)
Value-to-Weight: 1.32 from 1.18 US$ per kg (+11.4%)
Average distance to market (volume-weighted): 1835.0 from 1448.2 nm (+26.7%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    273  17877558.0  12656226.2      7.0      5.9
1        Meat and Prep     51   8724910.2   4181993.4      3.4      1.9
2           Vegetables    154   7497744.9   8523707.8      2.9      3.9
3                Dairy     70   6825484.3   3965361.0      2.7      1.8
4               Fruits     81   2715629.9   2280486.3      1.1      1.1
5        Fats and Oils     48   2341391.4   1693666.7      0.9      0.8
6      Sugar and Honey     59   2229710.2   3076210.9      0.9      1.4
7  Unprocessed Cereals     15     77872.2    315680.4      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,2846299.6,4925379.8,3106086.3,2901002.8,2909716.9,2983686.0,2644409.7,5799399.7,5496775.2,5581540.6,3805717.6,3590459.2,2501698.0,4573885.0,4643359.9,5634299.9,4683631.0,3721582.8
1,Fats and Oils,1279302.3,2162619.3,1388536.4,1295461.1,1298892.6,1336974.5,1193856.8,2457877.0,2365846.0,2412101.1,1679615.9,1587907.5,1134197.0,2009998.4,2044809.8,2471356.7,2059749.9,1641371.1
2,Fruits,2235212.3,3943251.4,2459913.0,2286292.8,2292694.8,2350694.2,2070167.6,4535237.4,4328058.0,4373197.9,3013128.2,2841794.9,1954792.0,3636342.3,3697008.4,4590942.0,3793023.0,2947871.6
3,Meat and Prep,5859902.3,10384917.5,6463455.0,6000807.9,6017111.8,6167836.5,5422808.7,11916616.9,11385090.7,11483271.0,7922809.5,7468301.0,5117700.0,9563656.4,9721727.8,12125846.3,10017659.8,7746014.8
4,Processed Cereals,7190157.5,12505609.1,7863137.2,7325944.5,7347556.5,7544571.6,6675254.5,14443402.6,13763330.0,13966444.1,9640959.0,9067140.5,6315020.6,11614620.4,11843579.9,14501046.1,12001453.3,9394792.3
5,Sugar and Honey,2012782.7,3376596.0,2166675.9,2032321.8,2039158.5,2099333.5,1880191.8,3940851.0,3739314.8,3840467.7,2643643.0,2488867.9,1786759.0,3173388.7,3244381.8,3841244.2,3188956.7,2576833.2
6,Unprocessed Cereals,171591.5,287351.6,184663.3,173024.4,173587.4,179042.7,160282.0,334811.4,318341.3,326412.3,227617.0,211676.7,152378.8,272701.7,280850.7,330981.3,275593.7,218359.6
7,Vegetables,8246480.5,14493342.9,9058045.4,8426279.3,8450369.9,8666303.0,7642987.2,16700333.1,15923757.2,16112591.5,11089200.1,10462617.1,7219978.8,13380984.1,13607324.8,16831364.6,13906294.6,10853091.4


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.30,0.04,0.01,0.01,0.02,0.46,0.42,0.43,0.17,0.14,0.30,0.31,0.46,0.32,0.18
1,Fats and Oils,0.38,0.05,0.01,0.01,0.02,0.54,0.50,0.52,0.21,0.17,0.37,0.39,0.57,0.40,0.22
2,Fruits,0.63,0.08,0.02,0.02,0.04,0.91,0.83,0.85,0.35,0.28,0.62,0.64,0.97,0.68,0.37
3,Meat and Prep,0.52,0.07,0.02,0.02,0.04,0.74,0.68,0.69,0.29,0.23,0.51,0.53,0.80,0.56,0.30
4,Processed Cereals,0.30,0.04,0.01,0.01,0.02,0.43,0.40,0.41,0.17,0.13,0.30,0.31,0.46,0.32,0.17
5,Sugar and Honey,0.61,0.07,0.01,0.01,0.04,0.92,0.83,0.88,0.34,0.27,0.62,0.65,0.92,0.63,0.35
6,Unprocessed Cereals,1.49,0.17,0.02,0.03,0.10,2.24,2.03,2.13,0.86,0.66,1.55,1.65,2.29,1.58,0.85
7,Vegetables,0.83,0.11,0.02,0.03,0.06,1.21,1.10,1.13,0.46,0.38,0.82,0.85,1.28,0.89,0.48
